# Workflow Orchestration Agent (Supervisor)
### When the process spans multiple domains with different tools and knowledge

A supervisor agent receives a task, decomposes it into subtasks, plans the sequence, and assigns each subtask to the right specialist agent. Each specialist executes using its own tools, returns a structured result, and the supervisor synthesizes the final output.

What's less understood is the line between an orchestrator that coordinates and one that reasons. In some cases — high-volume, regulated, or deterministic workflows — explicit coordination is exactly what's needed, and AWS Step Functions is the right tool. But when the process is ambiguous, multi-domain, or requires adapting to what intermediate results reveal, an LLM-based supervisor agent is the better fit. It interprets the goal, decides how to decompose it, and replans when what it finds changes what needs to happen next.

**Use an LLM-based supervisor when:**
- Task decomposition requires judgment — the orchestrator decides how to split work based on what arrives, not a predefined structure
- Intermediate results change what happens next — the orchestrator replans rather than following a static path
- The process spans genuinely different domains (legal, financial, technical) where static routing rules are insufficient
- Final synthesis requires reasoning over multiple structured results, not just aggregation

**Use a deterministic orchestrator (e.g. AWS Step Functions) when:**
- The workflow involves high-volume, deterministic fan-out (e.g. processing contracts in parallel)
- Regulated processes require full execution history and compensating transaction logic
- The workflow definition must be auditable and versioned independently of the LLM

### Business Use Case: AnyComp Telecom Customer Churn Prevention

A high-value customer requests cancellation. The supervisor coordinates three specialists: Account Analyst, Retention Strategist, and Communication Specialist.

### Architecture

```
Cancel Request → Supervisor Agent
                      │
          ┌───────────┼────────────────┐
          ▼           ▼                ▼
   Account      Retention      Communication
   Analyst      Strategist     Specialist
          │           │                │
          └───────────┼────────────────┘
                      ▼
            Supervisor synthesizes → Final retention strategy
```


## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.36.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Setup: Mock Data

In [ ]:
import json

REGION = 'us-east-1'

# Customer data for churn prevention scenario
CUSTOMER = {
    "customer_id": "CUST-001",
    "name": "Maria Chen",
    "plan": "Premium",
    "monthly_bill": 95.00,
    "tenure_years": 5,
    "lifetime_value": 5700.00,
    "churn_risk": "high",
    "cancel_reason": "Too expensive compared to competitors",
    "usage": {
        "avg_data_gb": 35,
        "avg_calls_min": 120,
        "international_calls": True,
        "streaming_hours": 22,
        "hotspot_usage": True
    },
    "contact": {"email": "maria.chen@example.com", "preference": "email"}
}

RETENTION_OFFERS = [
    {"offer_id": "RET-20", "name": "20% discount for 6 months", "monthly_savings": 19.00, "duration_months": 6},
    {"offer_id": "RET-FREE", "name": "Free international calling add-on", "monthly_savings": 15.00, "duration_months": 12},
    {"offer_id": "RET-UPGRADE", "name": "Free upgrade to Ultra plan for 3 months", "monthly_savings": 25.00, "duration_months": 3},
]

CANCEL_REQUEST = (
    "Customer CUST-001 (Maria Chen) has requested to cancel their Premium plan. "
    "Reason: too expensive compared to competitors. "
    "She has been a customer for 5 years with a lifetime value of $5,700. "
    "Analyze her account, find the best retention offer, and draft a personalized retention message."
)

print('Mock data loaded')

## Implementation 1: Strands Agents

The supervisor is a Strands `Agent` with three `@tool` functions, each wrapping a specialist agent. The supervisor decides which specialist to call and in what order.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

# ── Specialist 1: Account Analyst ────────────────────────────────────────────
account_analyst = Agent(
    model=model,
    system_prompt=(
        "You are an account analyst at AnyComp Telecom. "
        "Given customer data, produce a concise account summary: tenure, value, "
        "usage patterns, and churn risk assessment."
    ),
)

@tool
def analyze_account(customer_data: str) -> str:
    """Analyze a customer account for churn risk and value.
    Args:
        customer_data: JSON string with customer account details
    """
    return str(account_analyst(f'Analyze this customer account:\n{customer_data}'))


# ── Specialist 2: Retention Strategist ───────────────────────────────────────
retention_strategist = Agent(
    model=model,
    system_prompt=(
        "You are a retention strategist at AnyComp Telecom. "
        "Given a customer profile and available offers, select the best retention offer "
        "and explain why it matches this customer's usage patterns."
    ),
)

@tool
def select_retention_offer(customer_profile: str, available_offers: str) -> str:
    """Select the best retention offer for a customer.
    Args:
        customer_profile: Customer account analysis
        available_offers: JSON list of available retention offers
    """
    prompt = f'Customer profile:\n{customer_profile}\n\nAvailable offers:\n{available_offers}\n\nSelect the best offer and explain why.'
    return str(retention_strategist(prompt))


# ── Specialist 3: Communication Specialist ───────────────────────────────────
comm_specialist = Agent(
    model=model,
    system_prompt=(
        "You are a communication specialist at AnyComp Telecom. "
        "Draft a warm, personalized retention message for a customer considering cancellation. "
        "Reference their specific usage, the offer selected for them, and their tenure."
    ),
)

@tool
def draft_retention_message(customer_name: str, offer_details: str, account_summary: str) -> str:
    """Draft a personalized retention message.
    Args:
        customer_name: Customer's name
        offer_details: The selected retention offer and rationale
        account_summary: Account analysis summary
    """
    prompt = f'Draft a retention message for {customer_name}.\nAccount: {account_summary}\nOffer: {offer_details}'
    return str(comm_specialist(prompt))


# ── Supervisor Agent ──────────────────────────────────────────────────────────
supervisor = Agent(
    model=model,
    system_prompt=(
        "You are the Churn Prevention Supervisor at AnyComp Telecom. "
        "When a customer requests cancellation, you coordinate three specialists:\n"
        "1. Call analyze_account with the customer data to get an account assessment\n"
        "2. Call select_retention_offer with the assessment and available offers\n"
        "3. Call draft_retention_message with the customer name, offer, and assessment\n"
        "4. Summarize the complete retention strategy\n\n"
        "Always run all three steps in order."
    ),
    tools=[analyze_account, select_retention_offer, draft_retention_message],
)

print("=" * 60)
print("STRANDS — Supervisor Agent")
print("=" * 60)
result = supervisor(
    CANCEL_REQUEST + f'\n\nCustomer data: {json.dumps(CUSTOMER)}\nAvailable offers: {json.dumps(RETENTION_OFFERS)}'
)
print(result)